# Day 14 · Borgesian map — The Tattered Map

**Question.** A rider who uses a wheelchair needs a map at almost 1:1: every curb ramp, every broken
slab, every pole in the way. How much of Chicago has such a map, and what does it show where it exists?

**Approach.** Project Sidewalk Chicago: volunteers walk streets in street-level imagery and label what
they see; other users (and an AI validator) confirm or reject each label. A street counts as **mapped**
once a volunteer has walked it. The plate draws every mapped street as a thin ink line on the city
(Borges' map the size of the Empire, surviving in tatters), then reads one of them at the scale of the
address grid: one mile of Cicero Avenue by its 800 address numbers.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Project Sidewalk labels and streets (cached, city slice saved to `data/raw/`), community areas, CTA bus stops |
| 3. Validate | Counts, label types, validation |
| 4. Transform | Mapped streets, community areas |
| 5. Analyze | Pace; the Cicero mile; the finding |
| 6. QA | Sensitivity |
| 7. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import re
import subprocess
import zipfile
from datetime import date, timedelta
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import shapely
import shapely.ops


from atlas.grid import to_address
from atlas.ledger import Ledger, write_plate

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('day14')
DAY = Path('.')
REPO = DAY.resolve().parents[1]
RAW = DAY / 'data' / 'raw'
PROC = DAY / 'data' / 'processed'
OUT = DAY / 'output'
for p in (RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)

PS = 'https://sidewalk-chicago.cs.washington.edu/v3/api'   # chicago.projectsidewalk.org redirects here
UTM = 26916                     # metres, for buffers and lengths
BAD = ['Obstacle', 'SurfaceProblem', 'NoCurbRamp', 'NoSidewalk']

## 2. Pull

The full Project Sidewalk export (about 230 MB of labels and 180 MB of streets, Chicago and suburbs) is
cached in `data/processed/` (not committed). The city's slice, the columns used here, is saved as dated
gzipped CSVs in `data/raw/`; re-running uses them.

In [ ]:
def pull(endpoint: str) -> Path:
    path = PROC / f'ps_{endpoint}.geojson'
    if not path.exists():
        r = requests.get(f'{PS}/{endpoint}', params={'filetype': 'geojson'}, timeout=900)
        r.raise_for_status()
        path.write_bytes(r.content)
        log.info('Pulled %s: %.0f MB', endpoint, path.stat().st_size / 1e6)
    return path

city = gpd.read_file(REPO / 'reference/raw/city_boundary_2026-11-01.geojson').to_crs(UTM)
CITY = city.union_all()

lab_raw = sorted(RAW.glob('ps_labels_city_*.csv.gz'))
st_raw = sorted(RAW.glob('ps_streets_city_*.csv.gz'))
if not lab_raw or not st_raw:
    pulled = date.today()
    feats = json.loads(pull('rawLabels').read_text())['features']
    L = pd.DataFrame([{**{k: f['properties'].get(k) for k in ('label_id', 'user_id', 'label_type', 'severity', 'correct',
                                                              'agree_count', 'disagree_count', 'time_created',
                                                              'image_capture_date', 'street_edge_id', 'street_side')},
                       'tags': '|'.join(f['properties'].get('tags') or []),
                       'lon': round(f['geometry']['coordinates'][0], 6), 'lat': round(f['geometry']['coordinates'][1], 6)}
                      for f in feats])
    del feats
    g = gpd.GeoSeries.from_xy(L.lon, L.lat, crs=4326).to_crs(UTM)
    L = L[g.within(CITY).to_numpy()].copy()
    L['user'] = pd.factorize(L.user_id)[0]          # volunteers as numbers, not ids
    L['time_created'] = L.time_created.str[:10]
    L.drop(columns='user_id').to_csv(RAW / f'ps_labels_city_{pulled}.csv.gz', index=False)
    S = gpd.read_file(pull('streets'), columns=['street_edge_id', 'way_type', 'status', 'audit_count', 'label_count',
                                                 'user_count', 'first_label_date', 'last_label_date'])
    S = S[S.to_crs(UTM).intersects(CITY).to_numpy()].copy()
    S['wkt'] = shapely.to_wkt(S.geometry.to_numpy(), rounding_precision=6)
    pd.DataFrame(S.drop(columns='geometry')).to_csv(RAW / f'ps_streets_city_{pulled}.csv.gz', index=False)
    lab_raw = sorted(RAW.glob('ps_labels_city_*.csv.gz'))
    st_raw = sorted(RAW.glob('ps_streets_city_*.csv.gz'))
pulled = date.fromisoformat(lab_raw[-1].name.split('_')[-1][:10])
labels = pd.read_csv(lab_raw[-1])
streets = pd.read_csv(st_raw[-1])
streets = gpd.GeoDataFrame(streets, geometry=shapely.from_wkt(streets.wkt), crs=4326).drop(columns='wkt').to_crs(UTM)
log.info('Pulled %s: %d labels, %d street edges in the city', pulled, len(labels), len(streets))

# community areas (for hover and the findings), CTA bus stops
ca_path = RAW / 'community_areas.geojson'
if not ca_path.exists():
    ca_path.write_bytes(requests.get('https://data.cityofchicago.org/resource/igwz-8jzy.geojson', params={'$limit': 100}, timeout=120).content)
areas = gpd.read_file(ca_path).to_crs(UTM)[['community', 'area_numbe', 'geometry']].rename(columns={'community': 'name', 'area_numbe': 'num'})
gtfs_zip = sorted((REPO / 'reference/raw/gtfs').glob('cta_gtfs_*.zip'))[-1]
with zipfile.ZipFile(gtfs_zip) as z:
    bus_stops = pd.read_csv(z.open('stops.txt'))
gtfs_date = date.fromisoformat(gtfs_zip.stem.split('_')[-1])

## 3. Validate

In [ ]:
assert labels.label_id.is_unique and streets.street_edge_id.is_unique
assert set(labels.label_type) <= {'CurbRamp', 'NoCurbRamp', 'Obstacle', 'SurfaceProblem', 'Crosswalk', 'Signal', 'NoSidewalk', 'Occlusion', 'Other'}
assert len(areas) == 77
labels['state'] = np.select([labels.correct.eq(True) | labels.correct.eq('True'), labels.correct.eq(False) | labels.correct.eq('False')],
                            ['confirmed', 'rejected'], 'unchecked')
by_state = labels.state.value_counts()
print(by_state.to_dict())
print(pd.crosstab(labels.label_type, labels.state))

## 4. Transform

- **Mapped**: a street edge in the city with `status = open` (imagery exists) and at least one audit.
  Lengths in metres in UTM 16N.
- **Lines**: mapped edges merged into lines, simplified 3 m, in address units.
- **Community areas**: each edge counts in the area holding its midpoint.

In [ ]:
streets['km'] = streets.length / 1000
op = streets[streets.status.eq('open')].copy()
op['mapped'] = op.audit_count.gt(0)
open_km, mapped_km = op.km.sum(), op.loc[op.mapped, 'km'].sum()
print(f'open {open_km:,.0f} km, mapped {mapped_km:,.0f} km ({mapped_km / open_km:.1%}); no imagery {streets.loc[streets.status.eq("no_imagery"), "km"].sum():.0f} km')

def addr(geom_ll):
    a = np.asarray(geom_ll.coords)
    x, y = to_address(a[:, 1], a[:, 0])
    return [[int(round(i)), int(round(j))] for i, j in zip(x, y)]

merged = shapely.line_merge(shapely.union_all(op.loc[op.mapped].geometry.to_numpy()))
merged = gpd.GeoSeries([shapely.simplify(merged, 3)], crs=UTM).to_crs(4326).iloc[0]
lines = [addr(l) for l in getattr(merged, 'geoms', [merged]) if len(l.coords) >= 2]
print('mapped lines', len(lines), 'points', sum(len(l) for l in lines))

mid = gpd.GeoDataFrame(op[['km', 'mapped']], geometry=op.geometry.interpolate(0.5, normalized=True), crs=UTM)
mid = gpd.sjoin(mid, areas, predicate='within', how='left')
lab_pts = gpd.GeoDataFrame(labels, geometry=gpd.points_from_xy(labels.lon, labels.lat), crs=4326).to_crs(UTM)
lab_ca = gpd.sjoin(lab_pts[['label_type', 'state', 'geometry']], areas, predicate='within', how='left')
ca = mid.groupby('name').apply(lambda g: pd.Series({'km': g.km.sum(), 'mappedKm': g.loc[g.mapped, 'km'].sum()}), include_groups=False)
ca['share'] = ca.mappedKm / ca.km
conf = lab_ca[lab_ca.state.eq('confirmed')]
ca['labels'] = lab_ca.groupby('name').size()
ca['ramps'] = conf[conf.label_type.eq('CurbRamp')].groupby('name').size()
ca['barriers'] = conf[conf.label_type.isin(BAD)].groupby('name').size()
for col, t in [('obstacles', 'Obstacle'), ('surfaces', 'SurfaceProblem'), ('noRamp', 'NoCurbRamp'), ('noSidewalk', 'NoSidewalk')]:
    ca[col] = conf[conf.label_type.eq(t)].groupby('name').size()
ca = ca.fillna(0).sort_values('share')
area_rows = []
areas_ll = areas.to_crs(4326)
for _, a in areas_ll.iterrows():
    g = shapely.simplify(a.geometry, 0.0004)
    rings = [addr(p.exterior) for p in getattr(g, 'geoms', [g])]
    c = ca.loc[a['name']]
    area_rows.append(dict(name=a['name'].title().replace("'S", "'s"), num=int(a.num), rings=rings, km=round(float(c.km), 1),
                          share=round(float(c.share), 3), mappedKm=round(float(c.mappedKm), 1), labels=int(c.labels), ramps=int(c.ramps), barriers=int(c.barriers),
                          obstacles=int(c.obstacles), surfaces=int(c.surfaces), noRamp=int(c.noRamp), noSidewalk=int(c.noSidewalk)))
print(ca.share.describe().round(3).to_dict())

## 5. Analyze

**Pace.** Street length by the year of its first label (streets mapped without any label have no date).
The last twelve months set the pace for the rest.

**The Cicero mile.** Cicero Avenue (4800 W) from Madison to Chicago Avenue: the CTA bus stops at Madison
and Chicago (both directions, averaged) fix the two ends; the label's distance between them is scaled to
800 address numbers. The street's centre line drifts in address units across the mile, so it is fitted
through the bus stops at each crossing; a label west of it is on the west sidewalk.

In [ ]:
op['first'] = pd.to_datetime(op.first_label_date, errors='coerce', utc=True)
km_year = op[op.mapped].groupby(op.first.dt.year).km.sum()
year_ago = pd.Timestamp(pulled - timedelta(days=365), tz='UTC')
last_year_km = float(op.loc[op.mapped & op.first.ge(year_ago), 'km'].sum())
years_left = (open_km - mapped_km) / last_year_km
print({int(k): round(v) for k, v in km_year.items()}, 'last 12 months', round(last_year_km), 'km; years left', round(years_left, 1))
volunteers = labels.user.nunique()
first_year = int(op.first.dt.year.min())
img = pd.to_datetime(labels.image_capture_date, errors='coerce')
made = pd.to_datetime(labels.time_created, errors='coerce')
img_age = ((made - img).dt.days / 365.25).median()
img_years = img.dt.year.value_counts().sort_index()
print('volunteers', volunteers, 'first year', first_year, 'median image age at labelling (years)', round(img_age, 1))

# --- the Cicero mile ---
bx, by = to_address(bus_stops.stop_lat, bus_stops.stop_lon)
bus_stops['x'], bus_stops['y'] = bx, by
def pair(names):
    s = bus_stops[bus_stops.stop_name.isin(names) & bus_stops.x.between(-4950, -4750)]
    assert len(s) >= 2, names
    return float(s.x.mean()), float(s.y.mean())
y0 = pair(['Madison & Cicero'])[1]
y1 = pair(['Chicago & Cicero'])[1]
cross = {n: pair([f'Cicero & {n}']) for n in ['Washington', 'Maypole', 'Lake (Green Line)', 'Hubbard', 'Ohio', 'Huron', 'Chicago']}
cx_fit = np.polyfit([v[1] for v in cross.values()], [v[0] for v in cross.values()], 1)
num = lambda y: (y - y0) * 800 / (y1 - y0)
lx, ly = to_address(labels.lat, labels.lon)
labels['ax'], labels['ay'] = lx, ly
labels['dx'] = labels.ax - np.polyval(cx_fit, labels.ay)
mile = labels[labels.dx.abs().lt(22) & labels.ay.between(y0 - 4, y1 + 4) & labels.state.ne('rejected')].copy()
mile['n'] = num(mile.ay).clip(0, 800)
mile['side'] = np.where(mile.dx < 0, 'W', 'E')
mile_conf = mile[mile.state.eq('confirmed')]
mile_counts = mile_conf.groupby(['label_type', 'side']).size().unstack(fill_value=0)
print('mile labels', len(mile), 'confirmed', len(mile_conf)); print(mile_counts)
# every named street that meets the mile, from OpenStreetMap (the Geofabrik extract in reference/):
# arterials (primary, secondary, tertiary) and minor streets, placed where they meet its centre line
osm_raw = RAW / 'osm_cicero_mile_streets.geojson'
if not osm_raw.exists():
    pbf = sorted((REPO / 'reference/raw/osm').glob('illinois-*.osm.pbf'))[-1]
    a, b, c = PROC / 'cicero.osm.pbf', PROC / 'cicero_hw.osm.pbf', PROC / 'cicero_named.osm.pbf'
    subprocess.run(['osmium', 'extract', '-b', '-87.752,41.877,-87.738,41.899', str(pbf), '-o', str(a), '--overwrite'], check=True)
    subprocess.run(['osmium', 'tags-filter', str(a), 'w/highway', '-o', str(b), '--overwrite'], check=True)
    subprocess.run(['osmium', 'tags-filter', str(b), 'w/name', '-o', str(c), '--overwrite'], check=True)
    subprocess.run(['osmium', 'export', str(c), '-o', str(PROC / 'cicero.geojson'), '-f', 'geojson', '--overwrite'], check=True)
    gj = json.loads((PROC / 'cicero.geojson').read_text())
    keep = [f for f in gj['features'] if f['geometry']['type'] == 'LineString'
            and f['properties'].get('highway') in {'primary', 'secondary', 'tertiary', 'residential', 'unclassified'}]
    osm_raw.write_text(json.dumps({'type': 'FeatureCollection', 'source': f'OpenStreetMap contributors (ODbL), Geofabrik {pbf.name}',
                                   'features': [{'type': 'Feature', 'geometry': f['geometry'],
                                                 'properties': {k: f['properties'].get(k) for k in ('name', 'highway')}} for f in keep]}))
osm = json.loads(osm_raw.read_text())
ys = np.linspace(y0 - 40, y1 + 40, 50)
axis = shapely.LineString(np.c_[np.polyval(cx_fit, ys), ys])
y_at = lambda d: axis.interpolate(d).y
found = {}
for f in osm['features']:
    nm, hw = f['properties']['name'], f['properties']['highway']
    if 'Cicero' in nm:
        continue
    c = np.asarray(f['geometry']['coordinates'])
    fx_, fy_ = to_address(c[:, 1], c[:, 0])
    ln = shapely.LineString(np.c_[fx_, fy_])
    if ln.distance(axis) > 12:
        continue
    p_, _ = shapely.ops.nearest_points(axis, ln)
    t = float(num(p_.y))
    if -10 <= t <= 810:
        short = re.sub(r'^(West|East|North|South) ', '', nm)
        short = re.sub(r' (Street|Avenue|Boulevard|Road|Place|Court)$', '', short)
        found.setdefault(short, []).append((t, hw in {'primary', 'secondary', 'tertiary'}))
cross_rows = []
for nm, hits in found.items():
    t = float(np.mean([h[0] for h in hits]))
    t = 0.0 if nm == 'Madison' else 800.0 if nm == 'Chicago' else round(t, 1)
    cross_rows.append(dict(name=nm, n=t, major=any(h[1] for h in hits)))
cross_rows.sort(key=lambda r: r['n'])
print(cross_rows)
mile_km = (y1 - y0) / 800 * 1.609344

In [ ]:
share = mapped_km / open_km
finding = (f'Volunteers have mapped the sidewalks along {share:.0%} of Chicago’s streets, about one street in five, since {first_year}. '
           f'At the pace of the last year, mapping the rest would take about {round(years_left)} years.')
print(finding)

## 6. QA

In [ ]:
# 1. confirmed labels only vs everything not rejected, on the Cicero mile
print('mile: confirmed', len(mile_conf), 'not rejected', len(mile))
# 2. pace from the average of the last three full years instead of the last 12 months
full = km_year.loc[[pulled.year - 3, pulled.year - 2, pulled.year - 1]].mean()
print('years left at the 3-year average', round((open_km - mapped_km) / full, 1))
# 3. 'no imagery' streets are left out of the denominator; with them
print('share with no-imagery streets counted', round(mapped_km / (open_km + streets.loc[streets.status.eq('no_imagery'), 'km'].sum()), 3))
# 4. labels must sit on mapped streets
on = labels.street_edge_id.isin(op.loc[op.mapped, 'street_edge_id'])
print('labels on a mapped open street', f'{on.mean():.1%}')

## 7. Export

In [ ]:
pct_ = lambda v: round(float(v), 4)
lab_row = lambda r, extra: dict(t=r.label_type, ok={'confirmed': 1, 'unchecked': None}[r.state], sev=None if pd.isna(r.severity) else int(r.severity),
                                tags=[t for t in str(r.tags).split('|') if t and t != 'nan'][:3], d=str(r.time_created)[:7], **extra)
types = labels[labels.state.eq('confirmed')].label_type.value_counts()
summary = dict(
    openKm=round(open_km), mappedKm=round(mapped_km), share=pct_(share), noImageryKm=round(streets.loc[streets.status.eq('no_imagery'), 'km'].sum()),
    kmByYear={int(k): round(v) for k, v in km_year.items()}, lastYearKm=round(last_year_km), yearsLeft=round(years_left, 1),
    yearsLeft3=round(float((open_km - mapped_km) / full), 1), firstYear=first_year, volunteers=int(volunteers),
    labels=len(labels), confirmed=int(by_state.get('confirmed', 0)), rejected=int(by_state.get('rejected', 0)), unchecked=int(by_state.get('unchecked', 0)),
    types={k: int(v) for k, v in types.items()},
    noCurb=dict(confirmed=int(((labels.label_type == 'NoCurbRamp') & labels.state.eq('confirmed')).sum()),
                rejected=int(((labels.label_type == 'NoCurbRamp') & labels.state.eq('rejected')).sum())),
    imageAge=round(float(img_age), 1), imageYears=[int(img.dt.year.min()), int(img.dt.year.max())],
    areasHalf=int((ca.share > .5).sum()), areasTenth=int((ca.share < .1).sum()), areas=len(ca),
    mile=dict(labels=len(mile), confirmed=len(mile_conf), counts={t: {s: int(mile_counts.loc[t, s]) if s in mile_counts.columns else 0 for s in 'WE'} for t in mile_counts.index},
              km=round(float(mile_km), 2)),
)
plate = dict(
    summary=summary,
    lines=lines,
    areas=area_rows,
    mile=dict(labels=[lab_row(r, dict(n=round(float(r.n), 1), side=r.side)) for _, r in mile.sort_values('n').iterrows()], cross=cross_rows,
              ends=[[round(float(np.polyval(cx_fit, y)), 1), round(float(y), 1)] for y in (y0, y1)]),
    sources=dict(ps='https://sidewalk-chicago.cs.washington.edu/', api='https://sidewalk-chicago.cs.washington.edu/v3/api-docs/rawLabels',
                 areas='https://data.cityofchicago.org/d/igwz-8jzy',
                 gtfs='https://www.transitchicago.com/developers/gtfs/'),
    gtfsDate=gtfs_date.isoformat(),
)
ledger = Ledger(
    source='Project Sidewalk Chicago (University of Washington Makeability Lab)', dataset_id='v3/api rawLabels, streets',
    url='https://sidewalk-chicago.cs.washington.edu/', pulled=pulled, units='street km; labels',
    method=['a street is mapped once a volunteer has walked it in street-level imagery',
            'mapped streets drawn as thin lines'],
    caveats=['labels describe the imagery, not the street today',
             'missing-curb-ramp labels are often rejected in validation; only confirmed labels are counted'],
)
write_plate(OUT / 'plate.json', day=14, ledger=ledger, finding=finding, payload=plate)
print('plate.json', round((OUT / 'plate.json').stat().st_size / 1024), 'KB')